In [ ]:
import warnings
warnings.filterwarnings("ignore")
from biu_module import biuutilities as biu
engine = biu.create_alchemy_engine( "prod", "PROD_BIU_READ_ONLY_NS", '{"team":"biu-risk-analytics","owner":"70057258","job":"query","source":"jHub"}',)
conn = biu.get_sf_connection("prod", "PROD_BIU_READ_ONLY_NS", '{"team":"biu-risk-analytics","owner":"70057258","job":"query","source":"jHub"}',)
import pandas as pd
pd.options.display.max_columns=None
import numpy as np
import gc
import io
import boto3 
import pandas as pd
import pickle
import os
from datetime import datetime

import pandas as pd
import lightgbm as lgb
import glob
import pickle
import numpy as np
import math
import gc
import time 
import json

from biu_module import biuutilities
def get_snowflake_creds():
    user = sfOptions["sfUser"]
    password = sfOptions["sfPassword"]
    account = sfOptions["sfAccount"]
    database ='PCHFL_PROD' 
    warehouse = 'WH_BI_PROD' 
    role = 'PROD_BIU_READ_ONLY_NS' 
    schema = 'UNO_DS' 
    return user, password, account, database, warehouse, role, schema

connection = biuutilities.create_alchemy_engine(
    "prod",
    "PROD_BIU_READ_ONLY_NS",
    '{"team": "BIU", "owner": "70057258"}'
)

In [ ]:
def std_score(prob: int, pdo = 20, anchor = 600, odds_at_anchor = 50) -> int:
    """
    Converts model's probablity output to standardised score

    Parameters
    ----------
    prob           : Probability value from the model
    pdo            : Points for double odds
    anchor         : Reference score
    odds_at_anchor : Odds at the anchor

    Returns
    -------
    score : Score corresponding to the probability
    """

    factor = pdo / math.log(2, 10)
    offset = anchor - (factor*(math.log(odds_at_anchor, 10)))

    odds = prob / (1- prob)
    log_odds = math.log(odds, 10)
    
    score = offset - (factor*log_odds)

    return int(score)

In [ ]:
# Development Data Table (Snowflake):
# Purpose: Compute summary metrics.

# Responsibility: Model Owner

# Table Name Format: mm_dev_<model_name>
# (Example: uno_ds.mm_dev_hl_etc_v_2_3)

# Required Columns:

# Identifier : member_reference 

# Input features : ? 

# Output scores: raw_score, ventile/decile

# Sample Tag (development or OOT) : done 

# Development start date : source 



In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
# import xgboost as xgb

# MODEL_PATH = ".pkl"   # path to your trained XGBoost model
DEV_FILE = "dev 1.parquet"                   # dev dataset (with features, target, cust_id)
OOT_FILE = "oot 1.parquet"                   # oot dataset
OUTPUT_FILE = "mm_dev_bscore_offus_thick_v_1_0.parquet"   # final combined output

TARGET_COL = "target"
ID_COL = "member_reference"



# import joblib
# model = joblib.load(MODEL_PATH)

dev = pd.read_parquet(DEV_FILE)
oot = pd.read_parquet(OOT_FILE)

dev["sample_tag"] = "DEV"
oot["sample_tag"] = "OOT"


# feature_cols = [c for c in dev.columns if c not in [TARGET_COL, ID_COL]]
# dev["raw_score"] = model.predict_proba(dev[feature_cols])[:, 1]
# oot["raw_score"] = model.predict_proba(oot[feature_cols])[:, 1]


# dev["ventile"] = pd.qcut(dev["raw_score"], 20, labels=False, duplicates='drop') + 1
# oot["ventile"] = pd.qcut(oot["raw_score"], 20, labels=False, duplicates='drop') + 1




In [ ]:
gbm = lgb.Booster(model_file=f'model_2.4.7')
calibrator = pd.read_pickle('calibrator.pkl')


In [ ]:
df = dev.copy()
df['prob'] = gbm.predict(df[gbm.feature_name()])
df['raw_prob'] = gbm.predict(df[gbm.feature_name()],raw_score=True)
df['calib_prob'] = calibrator.predict_proba(df['raw_prob'].values.reshape(-1,1))[:,1]
df['model_score'] = df['calib_prob'].apply(std_score).clip(300,900)

df['ventile'] = pd.cut(df['model_score'],bins=[300,503,516,526,534,542,549,556,562,568,575,582,592,602,612,621,630,641,653,658,900],labels=['V'+str(i) for i in range(1,21)])

In [ ]:
df_oot = oot.copy()
df_oot['prob'] = gbm.predict(df_oot[gbm.feature_name()])
df_oot['raw_prob'] = gbm.predict(df_oot[gbm.feature_name()],raw_score=True)
df_oot['calib_prob'] = calibrator.predict_proba(df_oot['raw_prob'].values.reshape(-1,1))[:,1]
df_oot['model_score'] = df_oot['calib_prob'].apply(std_score).clip(300,900)

df_oot['ventile'] = pd.cut(df_oot['model_score'],bins=[300,503,516,526,534,542,549,556,562,568,575,582,592,602,612,621,630,641,653,658,900],labels=['V'+str(i) for i in range(1,21)])


In [ ]:
input_features = gbm.feature_name()
input_features

In [ ]:
df_oot = df_oot[['member_reference','no_of_active_trades_hl/no_of_active_trades_total',
 'no_of_enq_unsec_l12m',
 'g310s',
 'g406s',
 'fi29s',
 'fi31s',
 'no_of_ddup_enq_total_l9m',
 'amount_overdue_active_v12m',
 'fi34s',
 'no_of_enq_total_l6m',
 'months_since_5_plus_unsec',
 'br31s',
 'unsec_fraction_amount_of_trades_paid',
 'derog',
 'paymnt61',
 'paymnt62',
 's004s',
 'cc_fraction_count_of_trades_paid','model_score','ventile','sample_tag','source', 'target'
]]

In [ ]:
df = df[['member_reference','no_of_active_trades_hl/no_of_active_trades_total',
 'no_of_enq_unsec_l12m',
 'g310s',
 'g406s',
 'fi29s',
 'fi31s',
 'no_of_ddup_enq_total_l9m',
 'amount_overdue_active_v12m',
 'fi34s',
 'no_of_enq_total_l6m',
 'months_since_5_plus_unsec',
 'br31s',
 'unsec_fraction_amount_of_trades_paid',
 'derog',
 'paymnt61',
 'paymnt62',
 's004s',
 'cc_fraction_count_of_trades_paid','model_score','ventile','sample_tag','source','target'
]]
df.sample(5)

In [ ]:

final = pd.concat([df, df_oot], ignore_index=True)
final.to_parquet(OUTPUT_FILE, index=False)


In [ ]:
# final.to_sql('mm_dev_bscore_offus_thick_v_1_0',connection,if_exists='replace',schema='UNO_DS',chunksize=16000,index=False)

In [ ]:
final = pd.read_parquet('mm_dev_bscore_offus_thick_v_1_0.parquet')
final.head(2)

In [ ]:
final.columns

In [ ]:
# scorecard

In [ ]:
query = """Select * from 
                           ( Select cust_id, member_reference, batch_id, "no_of_active_trades_hl/no_of_active_trades_total",
                            no_of_enq_unsec_l12m,     no_of_ddup_enq_total_l9m,derog,
                            amount_overdue_active_v12m,      no_of_enq_total_l6m,
                            months_since_5_plus_unsec,   unsec_fraction_amount_of_trades_paid,
                            cc_fraction_count_of_trades_paid,to_timestamp(snapshot_date) as snapshot_date,model_filter_1_active_present,model_filter_2_thick,model_filter_3_current     from
                            aiops.fs_offus_bureau_report
                            where  run_id = 'BSCORE_PIPELINE_20260312114650') a
                            left join
                            (Select "Member Reference" as mr, g310s,
                                g406s, fi34s, fi29s, br31s, s004s, paymnt61, fi31s,
                                paymnt62, batch_id as bi, 1 as cv_available from uno_ds.bureau_cv_variable) b
                                on a.member_reference=b.mr  
                                and a.batch_id = b.bi """

scorecard_thick_offus = biuutilities.read_pandas_sf(conn, query)

# Execute query
# df_new = biuutilities.read_pandas_sf(conn=engine, query=q_new)
scorecard_thick_offus.columns = scorecard_thick_offus.columns.str.lower()




In [ ]:
scorecard_thick_offus = scorecard_thick_offus[['cust_id','member_reference', 'no_of_active_trades_hl/no_of_active_trades_total',
       'no_of_enq_unsec_l12m', 'g310s', 'g406s', 'fi29s', 'fi31s',
       'no_of_ddup_enq_total_l9m', 'amount_overdue_active_v12m', 'fi34s',
       'no_of_enq_total_l6m', 'months_since_5_plus_unsec', 'br31s',
       'unsec_fraction_amount_of_trades_paid', 'derog', 'paymnt61', 'paymnt62',
       's004s', 'cc_fraction_count_of_trades_paid']]

In [ ]:
# Output
print("Unique accounts:", scorecard_thick_offus['member_reference'].nunique())

print("Shape:", scorecard_thick_offus.shape)
(scorecard_thick_offus.sample(5))



In [ ]:
#  the actual output 
def pull_model_out1(model_out_table):

    ## Function to pull all on-us models output

    q = f"""
        SELECT *
        FROM PCHFL_PROD.AIOPS.{model_out_table}
        WHERE run_id = 'BSCORE_PIPELINE_20260312114650' and score !='-1' 

    """

    df = biuutilities.read_pandas_sf(conn, q)

    df.columns = df.columns.str.lower()

    print(f"Fetched {len(df)} rows.")
    print(f"Fetched {(df['cust_id'].nunique())} rows.")
    df.drop_duplicates(subset='cust_id',inplace=True)
#     print(f"Fetched {len(df)} rows.")
    return df
onus_final_model = 'FS_META_MODEL'        
onus_final_model1 = pull_model_out1(onus_final_model)  

In [ ]:
onus_final_model1['model_name'].value_counts(dropna=False)

In [ ]:
# onus_final_model1[onus_final_model1['model_name']=='thin_meta']

In [ ]:
thick_offus = onus_final_model1[onus_final_model1['model_name']=='thick_offus']

In [ ]:
print(f"Fetched {(thick_offus['cust_id'].nunique())} rows.")

In [ ]:
thick_offus = thick_offus.merge(scorecard_thick_offus , on='cust_id',how='left')
thick_offus

In [ ]:
thick_offus = thick_offus.sort_values('score', ascending=False).drop_duplicates(subset='cust_id', keep='first')

In [ ]:
thick_offus['member_reference'].nunique()

In [ ]:
thick_offus.shape

In [ ]:
final_scorecard = thick_offus[['cust_id','member_reference', 'no_of_active_trades_hl/no_of_active_trades_total',
       'no_of_enq_unsec_l12m', 'g310s', 'g406s', 'fi29s', 'fi31s',
       'no_of_ddup_enq_total_l9m', 'amount_overdue_active_v12m', 'fi34s',
       'no_of_enq_total_l6m', 'months_since_5_plus_unsec', 'br31s',
       'unsec_fraction_amount_of_trades_paid', 'derog', 'paymnt61', 'paymnt62',
       's004s', 'cc_fraction_count_of_trades_paid','score',	'decile',	'model_name',	'batch_id',	'run_id']]

In [ ]:
final_scorecard['batch_id'].value_counts()


In [ ]:
final_scorecard.rename(columns={'no_of_active_trades_hl/no_of_active_trades_total':'no_of_active_trades_hl_by_no_of_active_trades_total'} ,inplace=True)

In [ ]:
final_scorecard

In [ ]:
final_scorecard.to_parquet('thick_offus_scorecard_nov25.parquet')

In [ ]:
final.rename(columns={'no_of_active_trades_hl/no_of_active_trades_total':'no_of_active_trades_hl_by_no_of_active_trades_total'} ,inplace=True)

In [ ]:
final.to_parquet('thick_offus.parquet')

In [ ]:
final = pd.read_parquet('thick_offus.parquet')
final.head(2)

In [ ]:
# final_thin = pd.read_csv('mm_dev_bscore_offus_thin_v_1_0.csv')
# final_thin.head(2)






query = """select *
from uno_ds.mm_dev_bscore_offus_thin_v_1_0 """

final_thin = biuutilities.read_pandas_sf(conn, query)

# Execute query
# df_new = biuutilities.read_pandas_sf(conn=engine, query=q_new)
final_thin.columns = final_thin.columns.str.lower()



In [ ]:
final.head(2)

In [ ]:
final_thin.rename(columns={'fin_reference':'member_reference'},inplace=True)

In [ ]:
final_combined = pd.concat([final_thin , final])
final_combined

In [ ]:
filtered_df = final_combined[
    ((final_combined["sample_tag"] == "DEV")) |
    ( (final_combined["sample_tag"] == "DEV"))
]

In [ ]:
import pandas as pd
import numpy as np

# -------------------------------
# 1️⃣ Filter DEV data
# -------------------------------
dev_df = filtered_df[filtered_df['sample_tag'].str.lower() == 'dev'].copy()

if dev_df.empty:
    raise ValueError("No DEV data found — check 'sample_tag'!")

# -------------------------------
# 2️⃣ Create BSCORE (Score Ranges)
# -------------------------------
def assign_score_band(score):
    if 300 <= score <= 540:
        return '300-540'
    elif 541 <= score <= 580:
        return '541-580'
    elif 581 <= score <= 600:
        return '581-600'
    elif 601 <= score <= 640:
        return '601-640'
    elif 641 <= score <= 680:
        return '641-680'
    elif 681 <= score <= 900:
        return '681-900'
    else:
        return 'Unknown'

dev_df['bscore'] = dev_df['model_score'].apply(assign_score_band)

# -------------------------------
# 3️⃣ Exclude non-feature columns
# -------------------------------
exclude_cols = ['member_reference', 'sample_tag', 'target', 'source', 'model_score', 'model_name','ventile']
# exclude_cols = ['member_reference', 'sample_tag', 'target', 'source', 'model_score', 'model_name','ventile'
feature_cols = [col for col in dev_df.columns if col not in exclude_cols]

# -------------------------------
# 4️⃣ Feature Description Mapping
# -------------------------------
feature_desc_map = {
    'cv13': 'Percentage of accounts ever delinquent',
    'mt36s': 'Months since most recent mortgage delinquency',
    'tw34s': 'Utilization for open two wheeler loan trades verified in past 12 months',
    'bscore': 'Model score bands'
}

# -------------------------------
# 5️⃣ Generate PSI-style table
# -------------------------------
def generate_meta_bins(df, features, n_bins=10, method='quantile'):
    records = []

    for col in features:
        try:
            s = df[col].dropna()

            # -------------------------------
            # 🔥 Special handling for BSCORE
            # -------------------------------
            if col == 'bscore':
                dist = s.value_counts(normalize=True)

                # maintain correct order
                order = ['300-540', '541-580', '581-600', '601-640', '641-680','681-900']
                dist = dist.reindex(order).dropna()

                for i, (b, pct) in enumerate(zip(dist.index, dist.values), start=1):
                    records.append({
                        'INDEX': b,   # ✅ score range here
                        'EXPECTED_PER': pct,
                        'FEATURE': 'bscore',
                        'RANK': i,
                        'FEATURE_DESCRIPTION': feature_desc_map.get(col, col),
                        'PROD_TYPE': 'null'
                    })
                continue

            # -------------------------------
            # Numeric features
            # -------------------------------
            if not np.issubdtype(s.dtype, np.number):
                continue

            if s.nunique() < 2:
                continue

            if method == 'equal':
                bins = pd.cut(s, bins=n_bins, duplicates='drop')
            else:
                bins = pd.qcut(s, q=n_bins, duplicates='drop')

            bin_counts = bins.value_counts().sort_index()
            bin_percents = bin_counts / bin_counts.sum()

            for i, (b, pct) in enumerate(zip(bin_counts.index.astype(str), bin_percents), start=1):
                records.append({
                    'INDEX': b,
                    'EXPECTED_PER': pct,
                    'FEATURE': col,
                    'RANK': i,
                    'FEATURE_DESCRIPTION': feature_desc_map.get(col, col),
                    'PROD_TYPE': 'null'
                })

        except Exception as e:
            print(f"Skipping {col}: {e}")
            continue

    df_out = pd.DataFrame(records)

    if df_out.empty:
        raise ValueError("No valid features found!")

    # -------------------------------
    # 6️⃣ Scale RANK
    # -------------------------------
    df_out['RANK'] = df_out.groupby('FEATURE')['RANK'].transform(
        lambda x: x * (200 / x.max())
    )

    # -------------------------------
    # 7️⃣ Sort
    # -------------------------------
    df_out = df_out.sort_values(['FEATURE', 'RANK']).reset_index(drop=True)

    return df_out

# -------------------------------
# 8️⃣ Run
# -------------------------------
meta_bins_df = generate_meta_bins(dev_df, feature_cols)

# -------------------------------
# 9️⃣ Final column order
# -------------------------------
meta_bins_df = meta_bins_df[
    ['INDEX', 'EXPECTED_PER', 'FEATURE', 'RANK', 'FEATURE_DESCRIPTION', 'PROD_TYPE']
]

# -------------------------------
# 🔟 Save
# -------------------------------
meta_bins_df.to_csv("combined_offus_model_bins_dev.csv", index=False)

print("✅ Done! PSI table with SCORE RANGES (bscore) created.")
meta_bins_df.head()

In [ ]:
meta_bins_df.sort_values(by='RANK').head(40)

In [ ]:
meta_bins_df[meta_bins_df['FEATURE']=='bscore']

In [ ]:
# feature_bins_dev_df.to_sql('bscore_offus_thick_psi',connection,if_exists='replace',schema='UNO_DS',chunksize=16000,index=False)

In [ ]:
final_scorecard

In [ ]:
import json

metadata = {
  "MODEL_OWNER": ["abhishek.gupta6@piramal.com"],
  "DESCRIPTION": "COMBINED OFF US B SCORE MODEL",
  "MODEL_NAME": "COMBINED OFFUS",
  "ACTIVE_VERSION": "V1",
  "VERSIONS": {
    "MONITOR_FLAG": True,
    "V1": {
      "LIVE_DATE": "2025-11-11",
      "L2_LAYER_OP_REQ": True,
      "SUMMARY_CONFIG": {
        "table": "uno_ds.",
        "target_col": "target",
        "gini": -1,
        "prob_col": "",
        "id_col": "cust_id",
        "flag_col": "sample_tag",
        "excluded_ventiles": [],
        "compute_gini": True
      },
      "SCORECARD": None,
      "S3_PATH": "",
      "SCORECARD_TABLE_QUERY": """SELECT 
    cust_id,
    bscore,
    decile,
    model_name,
    batch_id,
    run_id,

    member_reference,
    at12s,
    at57s,
    bi36s,
    br31s,
    cvscore,
    fi29s,
    g051s,
    g235s,
    g310s,
    g406s,
    in36s,
    miss_pymnt_cnt_total_ratio,
    p_accounts_ever_5_plus,
    paymnt58,
    paymnt62,
    trv01,
    unsecured_amount_overdue_sum,
    fin_reference,

    NULL AS no_of_active_trades_hl_by_no_of_active_trades_total,
    NULL AS no_of_enq_unsec_l12m,
    NULL AS fi31s,
    NULL AS no_of_ddup_enq_total_l9m,
    NULL AS amount_overdue_active_v12m,
    NULL AS fi34s,
    NULL AS no_of_enq_total_l6m,
    NULL AS months_since_5_plus_unsec,
    NULL AS unsec_fraction_amount_of_trades_paid,
    NULL AS derog,
    NULL AS paymnt61,
    NULL AS s004s,
    NULL AS cc_fraction_count_of_trades_paid

FROM uno_ds.thin_offus_scorecard

UNION ALL

SELECT 
    cust_id,
    bscore,
    decile,
    model_name,
    batch_id,
    run_id,

    member_reference,
    NULL AS at12s,
    NULL AS at57s,
    NULL AS bi36s,
    br31s,
    NULL AS cvscore,
    fi29s,
    NULL AS g051s,
    NULL AS g235s,
    g310s,
    g406s,
    NULL AS in36s,
    NULL AS miss_pymnt_cnt_total_ratio,
    NULL AS p_accounts_ever_5_plus,
    NULL AS paymnt58,
    paymnt62,
    NULL AS trv01,
    NULL AS unsecured_amount_overdue_sum,
    NULL AS fin_reference,

    no_of_active_trades_hl_by_no_of_active_trades_total,
    no_of_enq_unsec_l12m,
    fi31s,
    no_of_ddup_enq_total_l9m,
    amount_overdue_active_v12m,
    fi34s,
    no_of_enq_total_l6m,
    months_since_5_plus_unsec,
    unsec_fraction_amount_of_trades_paid,
    derog,
    paymnt61,
    s004s,
    cc_fraction_count_of_trades_paid

FROM thick_offus_scorecard;""",
      "MASTER_PSI_TABLE": "uno_ds.combined_offus_model_bins_dev",
      "DEPENDENT_TABLES": [],
      "CSI_INDEX_TYPE": "numeric",
      "CSI_COLUMNS": ['no_of_active_trades_hl_by_no_of_active_trades_total',
       'no_of_enq_unsec_l12m', 'g310s', 'g406s', 'fi29s', 'fi31s',
       'no_of_ddup_enq_total_l9m', 'amount_overdue_active_v12m', 'fi34s',
       'no_of_enq_total_l6m', 'months_since_5_plus_unsec', 'br31s',
       'unsec_fraction_amount_of_trades_paid', 'derog', 'paymnt61', 'paymnt62',
       's004s', 'cc_fraction_count_of_trades_paid','at12s',
    'at57s',
    'bi36s',
    'br31s',
    'cvscore',
    'fi29s',
    'g051s',
    'g235s',
    'g310s',
    'g406s',
    'in36s',
    'miss_pymnt_cnt_total_ratio',
    'p_accounts_ever_5_plus',
    'paymnt58',
    'paymnt62',
    'trv01',
    'unsecured_amount_overdue_sum',
          
      ],
      "PREDICTED_MODEL_OUTPUT": ["bscore"],
      "IDENTIFIER_MAPPING": {
        "applicantId": "",
        "leadId": "cust_id",
        "timestamp": ""
      },
      "METRICS": {
        "PRESENT_IN_MODEL": ["bscore"],
        "METRICS_GENERATED": [],
        "GENERATED_METRICS_FUNCTION_REPO": {}
      },
      "EXTERNAL_SCORES": {
        "SCORE_NAME": {
          "IS_ACTIVE": False,
          "SOURCE_TYPE": "json-str",
          "SOURCE": "",
          "LEAD_ID_MAPPING": "cust_id"
        }
      },
      "I/O": {
        "INPUT_SCHEMA": ['cust_id','no_of_active_trades_hl_by_no_of_active_trades_total',
       'no_of_enq_unsec_l12m', 'g310s', 'g406s', 'fi29s', 'fi31s',
       'no_of_ddup_enq_total_l9m', 'amount_overdue_active_v12m', 'fi34s',
       'no_of_enq_total_l6m', 'months_since_5_plus_unsec', 'br31s',
       'unsec_fraction_amount_of_trades_paid', 'derog', 'paymnt61', 'paymnt62',
       's004s', 'cc_fraction_count_of_trades_paid','at12s',
    'at57s',
    'bi36s',
    'br31s',
    'cvscore',
    'fi29s',
    'g051s',
    'g235s',
    'g310s',
    'g406s',
    'in36s',
    'miss_pymnt_cnt_total_ratio',
    'p_accounts_ever_5_plus',
    'paymnt58',
    'paymnt62',
    'trv01',
    'unsecured_amount_overdue_sum',
          
        ],
        "OUTPUT_SCHEMA": [
          'cust_id',
          'bscore'
        ]
      }
    }
  }
}

with open("offus_combined.json", "w") as f:
    json.dump(metadata, f, indent=4)
